# **Google Drive**, **Directories** and **Imports**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
home_dir = '/content/drive/MyDrive/NUANS_Project'
dir_data= home_dir+'/BooksDataSet.csv'
dir_training = home_dir+'/Trainings'

dir_bert_tokenizer = home_dir+'/bert_tokenizer'
dir_bert_model = home_dir+'/bert_model'

In [ ]:
!pip install transformers

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import json
import pickle
from datetime import datetime, timedelta
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertModel, BertTokenizerFast, AdamW, get_linear_schedule_with_warmup #get_linear_schedule_with_warmup for a more dynamic learning rate schedule.
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm  # for progress bar
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
from torch.cuda.amp import autocast #used to integrate mixed precision training, useful to reduce memory usage and speed up training (16-bit floating-point numbers)

In [ ]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import nltk

# Download the WordNet resource for lemmatization
nltk.download('wordnet')

# Initialize spaCy and NLTK
nlp = spacy.load("en_core_web_sm")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# **Preprocessing** and **Dataset**

In [ ]:
# Load your data from the CSV file
data = pd.read_csv(dir_data)

# Extract text and labels
texts = data['summary'].values
labels = data['genre'].values

# Calculate the lengths of the summaries
summary_lengths = [len(summary.split()) for summary in texts]

# Calculate the number of bins and bin width
max_length = max(summary_lengths)
bin_width = 200
num_bins = max_length // bin_width + 1

plt.figure(figsize=(18, 6))
plt.hist(summary_lengths, bins=num_bins, range=(0, num_bins * bin_width), color='skyblue', edgecolor='black')
plt.title('Summary Length Distribution (200 Words per Bar)')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Set x-axis ticks to be in multiples of 200
plt.xticks(range(0, (num_bins + 1) * bin_width, bin_width))

plt.show()


In [ ]:
# Set the threshold (e.g., 1000 words)
threshold = 512

# Calculate the percentage of summaries with less than 1000 words
below_threshold = sum(1 for length in summary_lengths if length < threshold)
total_summaries = len(texts)
percentage_below_threshold = (below_threshold / total_summaries) * 100

print(f"Percentage of summaries with less than {threshold} words: {percentage_below_threshold:.2f}%")

In [ ]:
def preprocess_text(text):
    # Tokenize the text using spaCy
    doc = nlp(text)

    # Initialize an empty list to store modified tokens
    modified_tokens = []

    for token in doc:
        # Lowercase the token
        token_text = token.text.lower()

        if token_text not in STOP_WORDS:
            if token.ent_type_:
                # NER token, add START and END markers
                ner_type = token.ent_type_
                ner_text = token_text
                modified_tokens.append(f"START:{ner_type} {ner_text} END")
            else:
                # Non-NER token, lemmatize and stem it, then add it
                lemmatized = lemmatizer.lemmatize(token_text)
                stemmed = stemmer.stem(lemmatized)
                modified_tokens.append(stemmed)

    # Join the modified tokens back into a single string
    preprocessed_text = ' '.join(modified_tokens)

    return preprocessed_text

In [ ]:
class GenreClassifierDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': torch.tensor(label)
        }

# **Model** and **Trainer**

In [ ]:
# Define the GenreClassifierModel class
class GenreClassifierModel(torch.nn.Module):
    def __init__(self, num_labels, transformer_model, dropout_prob):
        super(GenreClassifierModel, self).__init__()
        self.bert = transformer_model
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.classifier = torch.nn.Linear(self.bert.config.hidden_size, num_labels)


    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        dropout_output = self.dropout(outputs.last_hidden_state[:, 0]) #outputs.last_hidden_state[:, 0] = embedding of the first token from the last layer of the BERT model (aggregate representation of the entire input sequence).
        logits = self.classifier(dropout_output)
        return logits

In [ ]:
# Define the GenreClassifierTrainer class (with validation phase)
class GenreClassifierTrainer():
    def __init__(self, model, train_dataset, val_dataset, test_dataset, batch_size, num_epochs, learning_rate, max_length, dropout_prob, gradient_accumulation_steps, weight_decay):
        self.model = model
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.test_dataset = test_dataset

        #Hyperparameters
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.learning_rate = learning_rate
        self.max_length = max_length
        self.dropout_prob = dropout_prob
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.weight_decay = weight_decay

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.best_val_accuracy = 0.0
        self.model_checkpoint_dir = None  # Path to the model checkpoint

        # Variables to track training and validation history
        self.train_loss_history = []
        self.train_accuracy_history = []
        self.val_loss_history = []
        self.val_accuracy_history = []

    def save_loss_accuracy_plots(self):
        # Add data points for the last epoch
        # These data points will represent the same accuracy and loss as the previous epoch
        self.train_loss_history.append(self.train_loss_history[-1])
        self.train_accuracy_history.append(self.train_accuracy_history[-1])
        # Add data points for the last epoch
        # These data points will represent the same accuracy and loss as the previous epoch
        self.val_loss_history.append(self.val_loss_history[-1])
        self.val_accuracy_history.append(self.val_accuracy_history[-1])

        plt.figure(figsize=(12, 5))

        # Plot training and validation loss
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss_history, label="Train Loss")
        plt.plot(self.val_loss_history, label="Validation Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Train and Validation Loss")
        plt.xlim(1, self.num_epochs)
        plt.xticks(range(1, self.num_epochs + 1))  # Show integer epochs
        plt.legend()
        for epoch, train_loss, val_loss in zip(range(1, self.num_epochs + 1), self.train_loss_history, self.val_loss_history):
          plt.text(epoch, train_loss, f"{train_loss:.2f}", ha='center', va='bottom')
          plt.text(epoch, val_loss, f"{val_loss:.2f}", ha='center', va='bottom')

        # Plot training and validation accuracy
        plt.subplot(1, 2, 2)
        plt.plot(self.train_accuracy_history, label="Train Accuracy")
        plt.plot(self.val_accuracy_history, label="Validation Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title("Train and Validation Accuracy")
        plt.xlim(1, self.num_epochs)
        plt.xticks(range(1, self.num_epochs + 1))  # Show integer epochs
        plt.yticks(np.arange(0, 1.1, 0.1)) # Show accuracy as real numbers from 0.0 to 1.0 with a scale of 0.1
        plt.legend()
        for epoch, train_acc, val_acc in zip(range(1, self.num_epochs + 1), self.train_accuracy_history, self.val_accuracy_history):
          plt.text(epoch, train_acc, f"{train_acc:.2f}", ha='center', va='bottom')
          plt.text(epoch, val_acc, f"{val_acc:.2f}", ha='center', va='bottom')

        # Save the plots
        plot_path = os.path.join(self.model_checkpoint_dir, "loss_accuracy_plots.png")
        plt.tight_layout()
        plt.savefig(plot_path)
        print(f"Loss and accuracy plots saved at {plot_path}")
        plt.show()


    def evaluate_and_save_best_model(self, val_labels, val_predictions, epoch):

        val_accuracy = accuracy_score(val_labels, val_predictions)

        # Save the model if validation accuracy is better
        if val_accuracy > self.best_val_accuracy:
            print("New Best Valid Accuracy found! Saving best model...")
            # Save accuracy and model
            self.best_val_accuracy= val_accuracy
            val_acc_info = f"Validation Accuracy = {val_accuracy:.4f}, reached at Epoch {epoch}/{self.num_epochs}."
            val_acc_info_path = os.path.join(self.model_checkpoint_dir, "val_acc_info.txt")
            with open(val_acc_info_path, "w") as file:
                file.write(val_acc_info)

            model_path = os.path.join(self.model_checkpoint_dir, "best_model.pth")
            torch.save(self.model.state_dict(), model_path)
            print("Best model saved!")

        return val_accuracy


    def train_model(self):
        # Get the current time
        current_time = datetime.now()
        # Add 2 hours to the current time
        adjusted_time = current_time + timedelta(hours=2)
        # Format the adjusted time as a string in the desired format
        adjusted_time_str = adjusted_time.strftime("%Y-%m-%d_%H:%M")
        self.model_checkpoint_dir = home_dir+"/Trainings/" + adjusted_time_str
        os.makedirs(self.model_checkpoint_dir, exist_ok=True)

        # Save hyperparameters to a JSON file
        hyperparameters = {
            "batch_size": self.batch_size,
            "num_epochs": self.num_epochs,
            "learning_rate": self.learning_rate,
            "max_length": self.max_length,
            "dropout_prob": self.dropout_prob,
            "gradient_accumulation_steps": self.gradient_accumulation_steps,
            "weight_decay": self.weight_decay
        }

        with open(os.path.join(self.model_checkpoint_dir, "hyperparameters.json"), "w") as json_file:
            json.dump(hyperparameters, json_file)

        optimizer = AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        scaler = torch.cuda.amp.GradScaler()  # For mixed precision training
        train_dataloader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

        for epoch in range(self.num_epochs):
            start_time = time.time()  # Record the start time for the epoch
            self.model.train()
            total_loss = 0
            correct_predictions = 0
            total_samples = 0

            progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f'Epoch {epoch + 1}/{self.num_epochs}')

            for i, batch in progress_bar:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                with autocast():  # Automatic mixed-precision
                    outputs = self.model(input_ids, attention_mask=attention_mask)
                    loss = torch.nn.functional.cross_entropy(outputs, labels)
                    loss = loss / self.gradient_accumulation_steps

                    predicted_labels = torch.argmax(outputs, dim=1)
                    correct_predictions += (predicted_labels == labels).sum().item()
                    total_samples += labels.size(0)

                scaler.scale(loss).backward()

                if (i + 1) % self.gradient_accumulation_steps == 0:
                    scaler.step(optimizer)
                    scaler.update()

                total_loss += loss.item()

                progress_bar.set_postfix(loss=f'{total_loss / (i + 1):.4f}', accuracy=f'{correct_predictions / total_samples:.4f}')

            # Calculate estimated time remaining
            progress_bar.close()

            # Calculate time taken for training
            time_taken = time.time() - start_time

            print(f"Epoch {epoch + 1} - Avg. Train Loss: {total_loss / len(train_dataloader):.4f}, Train Accuracy: {correct_predictions / total_samples:.4f}, Time Taken: {time_taken:.2f} seconds")
            # Append training loss and accuracy values to the history lists
            self.train_loss_history.append(total_loss / len(train_dataloader))
            self.train_accuracy_history.append(correct_predictions / total_samples)

            # After each epoch, evaluate on the validation set
            val_predictions, val_labels, val_loss = self.evaluate_model(self.val_dataset, True)
            #compute validation accuracy, the metric used to eventually save the best model
            val_accuracy = self.evaluate_and_save_best_model(val_labels, val_predictions, epoch+1)
            print(f"Epoch {epoch + 1} - Avg. Valid Loss: {val_loss:.4f}, Valid Accuracy: {val_accuracy:.4f}\n")
            self.val_accuracy_history.append(val_accuracy)
            self.val_loss_history.append(val_loss)

    def evaluate_model(self, dataset, valid = False):
        self.model.eval()
        dataloader = DataLoader(dataset, batch_size=self.batch_size)
        predictions = []
        true_labels = []
        total_loss = 0

        for batch in dataloader:
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels = batch['label'].to(self.device)
            with torch.no_grad():
                outputs = self.model(input_ids, attention_mask)

                if valid: #compute loss for plot analysis
                  loss = torch.nn.functional.cross_entropy(outputs, labels)
                  total_loss += loss.item()

                predicted_labels = torch.argmax(outputs, dim=1)
                predictions.extend(predicted_labels.to('cpu'))
                true_labels.extend(labels.to('cpu'))

        if valid:
          average_loss = total_loss / len(dataloader)
          return predictions, true_labels, average_loss
        else:
          return predictions, true_labels

# **Training phase**

In [ ]:
# Split the dataset into training, validation, and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)
train_texts, val_texts, train_labels, val_labels = train_test_split(train_texts, train_labels, test_size=0.2, random_state=42)

# Initialize a BERT tokenizer
bert_model = BertModel.from_pretrained(dir_bert_model)
bert_tokenizer = BertTokenizerFast.from_pretrained(dir_bert_tokenizer)

# Encode labels
label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
val_labels_encoded = label_encoder.transform(val_labels)
test_labels_encoded = label_encoder.transform(test_labels)

Perform this to use preprocessed texts:

In [ ]:
max_length = 512
# Apply preprocessing
preprocessed_train_texts = [preprocess_text(text) for text in train_texts]
preprocessed_val_texts = [preprocess_text(text) for text in val_texts]
preprocessed_test_texts = [preprocess_text(text) for text in test_texts]

# Create GenreClassifierDataset instances for training, validation, and testing with preprocessed texts
train_dataset = GenreClassifierDataset(preprocessed_train_texts, train_labels_encoded, bert_tokenizer, max_length=max_length)
val_dataset = GenreClassifierDataset(preprocessed_val_texts, val_labels_encoded, bert_tokenizer, max_length=max_length)
test_dataset = GenreClassifierDataset(preprocessed_test_texts, test_labels_encoded, bert_tokenizer, max_length=max_length)

Otherwise, perform only the code below:

In [ ]:
# Create GenreClassifierDataset instances for training, validation, and testing with texts
train_dataset = GenreClassifierDataset(train_texts, train_labels_encoded, bert_tokenizer, max_length=max_length)
val_dataset = GenreClassifierDataset(val_texts, val_labels_encoded, bert_tokenizer, max_length=max_length)
test_dataset = GenreClassifierDataset(test_texts, test_labels_encoded, bert_tokenizer, max_length=max_length)

In [ ]:
# Initialize the GenreClassifierModel
dropout_prob = 0.3
num_labels = len(label_encoder.classes_)
model = GenreClassifierModel(num_labels, bert_model, dropout_prob)

In [ ]:
# Initialize the GenreClassifierTrainer with the validation set
batch_size = 16
num_epochs = 20
learning_rate = 2e-5
gradient_accumulation_steps = 2
weight_decay = 0.1
trainer = GenreClassifierTrainer(model, train_dataset, val_dataset, test_dataset, batch_size, num_epochs, learning_rate, max_length, dropout_prob, gradient_accumulation_steps, weight_decay)

In [ ]:
# Train the model
trainer.train_model()
# After Training, save and show loss-accuracy plots
trainer.save_loss_accuracy_plots()

# **Testing phase**

To access the best_model, click on this link: https://drive.google.com/drive/folders/1trkEH5QEpaKgd3bC0UT35NLI_ViWMvjM?usp=sharing

In [ ]:
# Choose here the Training Experiment to use:
dir_exp = dir_training+"/2023-10-20_19:52"

In [ ]:
# Load hyperparameters from the JSON file
hyperparameters_file = os.path.join(dir_exp, "hyperparameters.json")
with open(hyperparameters_file, "r") as json_file:
    hyperparameters = json.load(json_file)

# Extract hyperparameters
batch_size = hyperparameters["batch_size"]
num_epochs = hyperparameters["num_epochs"]
learning_rate = hyperparameters["learning_rate"]
max_length = hyperparameters["max_length"]
dropout_prob = hyperparameters["dropout_prob"]
gradient_accumulation_steps = hyperparameters["gradient_accumulation_steps"]
weight_decay = hyperparameters["weight_decay"]

print("Batch Size:", batch_size)
print("Number of Epochs:", num_epochs)
print("Learning Rate:", learning_rate)
print("Max Length:", max_length)
print("Dropout Probability:", dropout_prob)
print("Gradient Accumulation Steps:", gradient_accumulation_steps)
print("Weight Decay:", weight_decay)

In [ ]:
# Read validation accuracy info
val_acc_info_path = os.path.join(dir_exp, "val_acc_info.txt")

with open(val_acc_info_path, "r") as file:
    val_acc_info_content = file.read()

print(val_acc_info_content)

In [ ]:
# Load your data from the CSV file
data = pd.read_csv(dir_data)

# Extract text and labels
texts = data['summary'].values
labels = data['genre'].values



*   Bert Model: https://drive.google.com/drive/folders/14rjclH6iI6cszPI84_XkWoUOBteIzl_N?usp=sharing
*   Bert Tokenizer: https://drive.google.com/drive/folders/1-BdEieeKH0sQrqyvKBX4_GP8BGcOyyZb?usp=sharing



In [ ]:
# Split the dataset into training, validation, and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)
train_texts, val_texts, train_labels, val_labels = train_test_split(train_texts, train_labels, test_size=0.2, random_state=42)

# Initialize a BERT tokenizer
bert_model = BertModel.from_pretrained(dir_bert_model)
bert_tokenizer = BertTokenizerFast.from_pretrained(dir_bert_tokenizer)

# Encode labels
label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
val_labels_encoded = label_encoder.transform(val_labels)
test_labels_encoded = label_encoder.transform(test_labels)

Perform this to use preprocessed texts:

In [ ]:
# Apply preprocessing
preprocessed_train_texts = [preprocess_text(text) for text in train_texts]
preprocessed_val_texts = [preprocess_text(text) for text in val_texts]
preprocessed_test_texts = [preprocess_text(text) for text in test_texts]
# Create GenreClassifierDataset instances for training, validation, and testing with preprocessed texts
train_dataset = GenreClassifierDataset(preprocessed_train_texts, train_labels_encoded, bert_tokenizer, max_length=max_length)
val_dataset = GenreClassifierDataset(preprocessed_val_texts, val_labels_encoded, bert_tokenizer, max_length=max_length)
test_dataset = GenreClassifierDataset(preprocessed_test_texts, test_labels_encoded, bert_tokenizer, max_length=max_length)

Otherwise, perform only the code below:

In [ ]:
# Create GenreClassifierDataset instances for training, validation, and testing with texts
train_dataset = GenreClassifierDataset(train_texts, train_labels_encoded, bert_tokenizer, max_length=max_length)
val_dataset = GenreClassifierDataset(val_texts, val_labels_encoded, bert_tokenizer, max_length=max_length)
test_dataset = GenreClassifierDataset(test_texts, test_labels_encoded, bert_tokenizer, max_length=max_length)

In [ ]:
# Initialize the GenreClassifierModel
num_labels = len(label_encoder.classes_)
model = GenreClassifierModel(num_labels, bert_model, dropout_prob)

In [ ]:
# Load the best model weights
model.load_state_dict(torch.load(os.path.join(dir_exp, "best_model.pth")))

In [ ]:
# Initialize the GenreClassifierTrainer with the validation set
trainer = GenreClassifierTrainer(model, train_dataset, val_dataset, test_dataset, batch_size, num_epochs, learning_rate, max_length, dropout_prob, gradient_accumulation_steps, weight_decay)

In [ ]:
# Evaluate the model on the test set
test_predictions, test_true_labels = trainer.evaluate_model(test_dataset)

In [ ]:
# Decode labels for the test set
predicted_labels = label_encoder.inverse_transform(test_predictions)
true_labels = label_encoder.inverse_transform(test_true_labels)

accuracy = accuracy_score(true_labels, predicted_labels)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

In [ ]:
report = classification_report(true_labels, predicted_labels)
print(report)

In [ ]:
# Compute the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Compute percentages
cm_percentages = cm / cm.sum(axis=1)[:, np.newaxis] * 100
fmt = lambda x: '{:.1f}%'.format(x)

class_names = ["Crime Fiction", "Fantasy", "Historical novel", "Horror", "Science Fiction", "Thriller"]
# Display the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_percentages, annot=True, fmt=".1f", cmap='Blues', annot_kws={'size': 10}, xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

# Save the image to the dir_exp directory
image_path = os.path.join(dir_exp, "confusion_matrix.png")
plt.savefig(image_path, format="png")
plt.show()
